# Offline GP logEI vs BNN logEI Replay Experiment

Replay optimization over the cached `ext/` BCLF Touchstone dataset. The campaign uses the same seeds, budget, initialization count, and BCLF objective settings as `notebooks/turbo_online_experiment.ipynb`, but it disables trust-region behavior and compares only two acquisition backends inside `OptimizerBoTorch`:

- `gp_qlogei`: GP logEI backend
- `bnn_qlogei`: scalar-objective BNN logEI backend

No HFSS queue or online simulator is used. The BNN logEI backend follows the few-shot training schedule: 100 epochs on the initial 8 evaluated samples, then warm-start updates for 10 epochs whenever one new evaluated sample is added.


In [ ]:
from __future__ import annotations

import json
import math
import sys
import time
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

def find_project_root(start: Path) -> Path:
    candidates = [start.resolve(), *start.resolve().parents, Path('/home/zhanghf/projects/vector')]
    for candidate in candidates:
        if (candidate / 'src' / 'optimization').exists() and (candidate / 'src' / 'modeling').exists():
            return candidate
    raise RuntimeError(f'Cannot locate project root from cwd={start.resolve()}')


PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
for path in (PROJECT_ROOT, SRC):
    path_str = str(path)
    if path_str in sys.path:
        sys.path.remove(path_str)
    sys.path.insert(0, path_str)

from modeling import BCLFTouchstoneDatasetLoader
from optimization import (
    BCLFOptimizationSpec,
    BNNExpectedImprovementBackend,
    GPqLogEIAcquisitionBackend,
    DiscreteCandidateSpace,
    ExperimentRecorder,
    OptimizerBoTorch,
    PassiveDataset,
    ReplaySimulator,
)


## Runtime Settings

Set `RUN_REPLAY=True` to execute the full campaign. The default is false so the notebook can be opened and validated without training BNNs.


In [ ]:
RUN_REPLAY = False
RESET_CHECKPOINTS = False
RESUME_IF_CHECKPOINT = True
FORCE_RERUN_COMPLETED = False

SEEDS = [0, 1, 2, 3, 4]
N_INIT = 8
BUDGET = 40
TARGET_TRIALS = N_INIT + BUDGET

DATASET_DIR = PROJECT_ROOT / 'data' / 'bandpass_coupled_line_filter_export' / 'ext'
OUTPUT_DIR = PROJECT_ROOT / 'experiments' / 'gp_qlogei_vs_bnn_qlogei_replay'
RUN_PREFIX = 'gp_qlogei_vs_bnn_qlogei_replay'

BNN_BACKEND_CONFIG = {
    'hidden_dims': [64, 64],
    'epochs': 100,
    'update_epochs': 10,
    'warm_start': True,
    'initial_train_size': N_INIT,
    'batch_size': 32,
    'lr': 1e-3,
    'bayesian_loss_weight': 1e-4,
    'mc_samples': 64,
    'acquisition': 'qlogei',
}
BNN_DEVICE = None  # None means CUDA if PyTorch sees it, otherwise CPU.

METHODS = {
    'gp_qlogei': {'label': 'GP logEI', 'backend': 'gp_qlogei'},
    'bnn_qlogei': {'label': 'BNN logEI', 'backend': 'bnn_qlogei'},
}

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('run_replay:', RUN_REPLAY)
print('dataset_dir:', DATASET_DIR)
print('output_dir:', OUTPUT_DIR)
print('seeds:', SEEDS)
print('target_trials:', TARGET_TRIALS)


## Dataset And Objective

The objective mirrors the current TuRBO online notebook settings documented for BCLF.


In [ ]:
response_dataset = BCLFTouchstoneDatasetLoader(dataset_dir=DATASET_DIR).load()

samples = []
for idx in range(response_dataset.n_samples):
    samples.append({
        'geo_params': response_dataset.X[idx],
        's_params': response_dataset.s_params[idx],
        'caseid': response_dataset.case_ids[idx],
        'filename': response_dataset.filenames[idx],
    })

dataset = PassiveDataset(
    samples=samples,
    variable_order=list(response_dataset.variable_order),
    freqs=response_dataset.freqs,
    device_name='bclf_ext_touchstone',
)

spec = BCLFOptimizationSpec(
    passband_ghz=(4.8, 5.4),
    lower_stopband_ghz=(3.6, 4.4),
    upper_stopband_ghz=(5.8, 8.7),
    min_passband_s21_db=-5.0,
    max_passband_s11_db=-7.0,
    max_stopband_s21_db=-15.0,
    s11_weight=0.5,
    stopband_weight=0.5,
)

print('samples:', len(dataset))
print('variables:', dataset.variable_order)
print('freq_points:', len(dataset.freqs), 'range_hz:', (float(dataset.freqs.min()), float(dataset.freqs.max())))
print(json.dumps(spec.describe(), indent=2))


## Campaign Helpers

In [ ]:
def method_backend(method: str):
    spec_info = METHODS[method]
    if spec_info['backend'] == 'bnn_qlogei':
        return BNNExpectedImprovementBackend(BNN_BACKEND_CONFIG, device=BNN_DEVICE)
    if spec_info['backend'] == 'gp_qlogei':
        return GPqLogEIAcquisitionBackend()
    return None


def run_id_for(method: str, seed: int) -> str:
    return f'{RUN_PREFIX}_{method}_seed{seed}'


def checkpoint_for(method: str, seed: int) -> Path:
    return OUTPUT_DIR / f'{run_id_for(method, seed)}.checkpoint.json'


def checkpoint_history(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        return []
    payload = json.loads(path.read_text(encoding='utf-8'))
    history = payload.get('history', [])
    return history if isinstance(history, list) else []


def checkpoint_completed(path: Path) -> bool:
    return len(checkpoint_history(path)) >= TARGET_TRIALS


def checkpoint_compatible(path: Path, method: str) -> bool:
    if not path.exists():
        return False
    payload = json.loads(path.read_text(encoding='utf-8'))
    order = payload.get('candidate_space', {}).get('variable_order', [])
    if list(order) != list(dataset.variable_order):
        return False
    run_id = str(payload.get('run_id', ''))
    return run_id == run_id_for(method, int(run_id.rsplit('seed', 1)[-1])) if 'seed' in run_id else False


def shuffled_indices(seed: int) -> np.ndarray:
    indices = np.arange(len(dataset), dtype=int)
    rng = np.random.default_rng(seed)
    rng.shuffle(indices)
    return indices


def make_candidate_space(seed: int) -> DiscreteCandidateSpace:
    indices = shuffled_indices(seed)
    geometries = np.asarray([dataset.get_geometry(int(i)) for i in indices], dtype=float)
    return DiscreteCandidateSpace(
        variable_order=dataset.variable_order,
        candidate_geometries=geometries,
        match_atol=dataset.match_atol,
        match_rtol=dataset.match_rtol,
    )


def build_optimizer(method: str, seed: int, *, resume: bool) -> OptimizerBoTorch:
    return OptimizerBoTorch(
        candidate_space=make_candidate_space(seed),
        simulator=ReplaySimulator(dataset=dataset),
        optimization_spec=spec,
        recorder=ExperimentRecorder(output_dir=OUTPUT_DIR),
        budget=BUDGET,
        n_init=N_INIT,
        seed=seed,
        run_id=run_id_for(method, seed),
        checkpoint_path=checkpoint_for(method, seed),
        resume=resume,
        acquisition_backend=method_backend(method),
    )


def summarize_history(method: str, seed: int, history: list[dict[str, Any]], status: str, runtime_sec: float) -> dict[str, Any]:
    objectives = [float(row.get('objective', float('inf'))) for row in history]
    finite = [v for v in objectives if np.isfinite(v)]
    best = min(finite) if finite else float('nan')
    cache_values = [row.get('cache_hit') for row in history if row.get('cache_hit') is not None]
    cache_hit_ratio = float(np.mean(cache_values)) if cache_values else float('nan')
    run_id = run_id_for(method, seed)
    return {
        'method': method,
        'method_label': METHODS[method]['label'],
        'seed': int(seed),
        'run_id': run_id,
        'status': status,
        'num_trials': int(len(history)),
        'target_trials': int(TARGET_TRIALS),
        'best_objective': float(best),
        'runtime_sec': float(runtime_sec),
        'cache_hit_ratio': cache_hit_ratio,
        'checkpoint_path': str(checkpoint_for(method, seed)),
        'csv_path': str(OUTPUT_DIR / f'{run_id}.csv'),
        'json_path': str(OUTPUT_DIR / f'{run_id}.json'),
    }


## Execute Campaign

In [ ]:
summary_rows = []
run_histories = []

if not RUN_REPLAY:
    print('RUN_REPLAY=False; skipping execution.')
else:
    for seed in SEEDS:
        for method in METHODS:
            ckpt = checkpoint_for(method, seed)
            if RESET_CHECKPOINTS and ckpt.exists():
                ckpt.unlink()
            completed = checkpoint_completed(ckpt)
            compatible = RESUME_IF_CHECKPOINT and ckpt.exists() and checkpoint_compatible(ckpt, method)
            if completed and not FORCE_RERUN_COMPLETED:
                history = checkpoint_history(ckpt)
                print(f'[{method}][seed={seed}] completed; skipping ({len(history)} trials)')
                summary_rows.append(summarize_history(method, seed, history, 'skipped_completed', 0.0))
                run_histories.extend(dict(row, method=method, method_label=METHODS[method]['label'], seed=seed, run_id=run_id_for(method, seed)) for row in history)
                continue

            print(f'[{method}][seed={seed}] running resume={compatible} checkpoint={ckpt.name}')
            start = time.perf_counter()
            optimizer = build_optimizer(method, seed, resume=compatible)
            result = optimizer.run()
            runtime = time.perf_counter() - start
            status = 'completed' if len(result.history) >= TARGET_TRIALS else 'partial'
            summary_rows.append(summarize_history(method, seed, result.history, status, runtime))
            run_histories.extend(dict(row, method=method, method_label=METHODS[method]['label'], seed=seed, run_id=result.run_id) for row in result.history)
            print(f'[{method}][seed={seed}] {status}: trials={len(result.history)} best={result.state.best_objective}')

summary_df = pd.DataFrame(summary_rows)
runs_df = pd.DataFrame(run_histories)
if len(summary_df):
    summary_df.to_csv(OUTPUT_DIR / 'comparison_summary.csv', index=False)
    display(summary_df)
if len(runs_df):
    runs_df.to_csv(OUTPUT_DIR / 'comparison_runs.csv', index=False)
    display(runs_df.tail())


## Load Existing Results

Use this cell after a previous campaign run to load CSV artifacts without rerunning the optimizers.


In [ ]:
summary_path = OUTPUT_DIR / 'comparison_summary.csv'
runs_path = OUTPUT_DIR / 'comparison_runs.csv'

if summary_path.exists():
    summary_df = pd.read_csv(summary_path)
else:
    summary_df = pd.DataFrame(summary_rows)

if runs_path.exists():
    runs_df = pd.read_csv(runs_path)
else:
    runs_df = pd.DataFrame(run_histories)

print('summary rows:', len(summary_df))
print('run rows:', len(runs_df))
if len(summary_df):
    display(summary_df)


## Plots

In [ ]:
if runs_df.empty:
    print('No run data available for plotting.')
else:
    plot_df = runs_df.copy()
    plot_df['iteration'] = pd.to_numeric(plot_df['iteration'], errors='coerce')
    plot_df['objective'] = pd.to_numeric(plot_df['objective'], errors='coerce')
    plot_df['best_so_far'] = pd.to_numeric(plot_df['best_so_far'], errors='coerce')
    plot_df = plot_df[np.isfinite(plot_df['iteration'])]

    fig, ax = plt.subplots(figsize=(9, 5))
    for method, label in [(m, METHODS[m]['label']) for m in METHODS]:
        mdf = plot_df[plot_df['method'] == method]
        for _, rdf in mdf.groupby('run_id'):
            rdf = rdf.sort_values('iteration')
            ax.plot(rdf['iteration'], rdf['best_so_far'], alpha=0.18, linewidth=1.0)
        if len(mdf):
            agg = mdf.groupby('iteration')['best_so_far'].quantile([0.25, 0.5, 0.75]).unstack()
            ax.plot(agg.index, agg[0.5], linewidth=2.5, label=label)
            ax.fill_between(agg.index, agg[0.25], agg[0.75], alpha=0.15)
    ax.set_title('Best Objective So Far by Method')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Best objective so far (lower is better)')
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'best_so_far_by_method.png', dpi=180)
    plt.show()

    fig, ax = plt.subplots(figsize=(9, 5))
    for method, label in [(m, METHODS[m]['label']) for m in METHODS]:
        mdf = plot_df[plot_df['method'] == method]
        if len(mdf):
            agg = mdf.groupby('iteration')['objective'].quantile([0.25, 0.5, 0.75]).unstack()
            ax.plot(agg.index, agg[0.5], linewidth=2.5, label=label)
            ax.fill_between(agg.index, agg[0.25], agg[0.75], alpha=0.15)
    ax.set_title('Objective by Iteration')
    ax.set_xlabel('Iteration')
    ax.set_ylabel('Objective (lower is better)')
    ax.grid(True, alpha=0.25)
    ax.legend()
    fig.tight_layout()
    fig.savefig(OUTPUT_DIR / 'objective_by_iteration.png', dpi=180)
    plt.show()

    if len(summary_df):
        fig, ax = plt.subplots(figsize=(7, 4))
        order = list(METHODS.keys())
        data = [summary_df.loc[summary_df['method'] == method, 'best_objective'].dropna().to_numpy() for method in order]
        ax.boxplot(data, labels=[METHODS[m]['label'] for m in order], showmeans=True)
        for pos, values in enumerate(data, start=1):
            if len(values):
                jitter = np.linspace(-0.06, 0.06, len(values)) if len(values) > 1 else np.asarray([0.0])
                ax.scatter(np.full(len(values), pos) + jitter, values, s=36, alpha=0.8)
        ax.set_title('Final Best Objective by Method')
        ax.set_ylabel('Best objective (lower is better)')
        ax.grid(True, axis='y', alpha=0.25)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / 'final_best_by_method.png', dpi=180)
        plt.show()


## Compact Summary

In [ ]:
if summary_df.empty:
    print('No summary data available.')
else:
    compact = summary_df.groupby(['method', 'method_label']).agg(
        completed=('status', lambda s: int(sum(str(v).startswith(('completed', 'skipped_completed')) for v in s))),
        runs=('run_id', 'count'),
        median_best=('best_objective', 'median'),
        mean_best=('best_objective', 'mean'),
        min_best=('best_objective', 'min'),
        median_trials=('num_trials', 'median'),
        total_runtime_sec=('runtime_sec', 'sum'),
    ).reset_index()
    display(compact)
